# 🌸 GlowAI — Data Cleaning Pipeline
**Notebook:** `01_data_cleaning.ipynb`  
**Purpose:** Load, inspect, clean, combine and export all raw datasets  

---

## Dataset Groups

| # | Group | Source Files | Output File |
|---|-------|-------------|-------------|
| 1 | Reviews (combined) | reviews_0-250, 250-500, 500-750, 750-1250, 1250-end, productReviews | `01_reviews_clean.csv` |
| 2 | Skincare Products | skincare_df, moisturizers, treatments, eyecare, wellness | `02_skincare_products_clean.csv` |
| 3 | Product URLs | product_urls, eyecare_url, treatment_urls, wellness_urls | `03_product_urls_clean.csv` |
| 4 | Popularity Rankings | most_expensive, most_popular_by_loves, _by_n_reviews, _by_score | `04_popularity_rankings_clean.csv` |
| 5 | Product Info | product_info | `05_product_info_clean.csv` |
| 6 | Product List | productlist | `06_productlist_clean.csv` |
| 7 | Sephora Products | Sephora_all_423 | `07_sephora_clean.csv` |
| 8 | Brands | brands_w_m_products | `08_brands_clean.csv` |
| 9 | Makeup (JSON) | makeup_data | `09_makeup_clean.csv` |
| 10 | Ingredients | binary_cosmetic_ingredient | `10_ingredients_clean.csv` |
| 11 | Paula — SUM LIST | Paula_SUM_LIST | `11_paula_sumlist_clean.csv` |
| 12 | Paula — Embeddings | Paula_embedding_SUMLIST_before_422 | `12_paula_embeddings_clean.csv` |
| 13 | Alternatives | pre_alternatives | `13_alternatives_clean.csv` |
| 14 | Datasheet | datasheet | `14_datasheet_clean.csv` |
| 15 | Output / Analysis | output, pretty_numbers, pretty_regression | `15_output_clean.csv`, `16_pretty_numbers_clean.csv`, `17_pretty_regression_clean.csv` |

---
## ⚙️ Section 0 — Setup, Imports & Configuration

In [50]:
import os
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

print('📍 Setting up paths for PUREGLOW AI structure...')
print(f'Current location: {Path.cwd()}')


📍 Setting up paths for PUREGLOW AI structure...
Current location: c:\Users\HP\OneDrive\Desktop\PureGlow AI\notebooks


In [51]:
RAW_DIR = Path('../data/raw').resolve()
OUTPUT_DIR = (Path('../data/cleaned')).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Input  : {RAW_DIR}')
print(f'Output : {OUTPUT_DIR}')
print(f'Exists : {RAW_DIR.exists()}')

if not RAW_DIR.exists():
    print('\n⚠️  ERROR: Could not find data/raw/')
    print('Make sure:')
    print('  1. Notebook is in: PUREGLOW AI/notebooks/data/')
    print('  2. CSV files are in: PUREGLOW AI/data/raw/')
    print('\nOr move the notebook to project root next to data/ folder')

Input  : C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\raw
Output : C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned
Exists : True


In [52]:
# ─── Helper Utilities ─────────────────────────────────────────────────────────

def load_file(filename: str, sheet_index: int = 0) -> pd.DataFrame:
    """Load an Excel file from RAW_DIR and return a DataFrame."""
    path = RAW_DIR / filename
    if not path.exists():
        print(f'  ⚠  File not found: {path}')
        return pd.DataFrame()
    df = pd.read_excel(path, sheet_name=sheet_index)
    print(f'  ✓  Loaded  {filename:<45} → {df.shape[0]:>8,} rows × {df.shape[1]} cols')
    return df



def load_file(filename: str, sheet_index: int = 0) -> pd.DataFrame:
    """Load a CSV or Excel file from RAW_DIR — auto-detects extension."""
    path = RAW_DIR / filename
    # If not found, try swapping extension
    if not path.exists():
        alt = path.with_suffix('.csv') if path.suffix in ('.xlsx', '.xls') else path.with_suffix('.xlsx')
        if alt.exists():
            path = alt
        else:
            print(f'  ⚠  File not found: {filename}')
            return pd.DataFrame()
    try:
        df = pd.read_csv(path) if str(path).endswith('.csv') else pd.read_excel(path, sheet_name=sheet_index)
        print(f'  ✓  Loaded  {path.name:<45} → {df.shape[0]:>8,} rows × {df.shape[1]} cols')
        return df
    except Exception as e:
        print(f'  ✗  Error reading {filename}: {e}')
        return pd.DataFrame()

def load_json(filename: str) -> pd.DataFrame:
    """Load a JSON file and normalise it into a flat DataFrame."""
    path = RAW_DIR / filename
    if not path.exists():
        print(f'  ⚠  File not found: {path}')
        return pd.DataFrame()
    with open(path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    # Handle both top-level list and top-level dict with a data key
    if isinstance(raw, list):
        df = pd.json_normalize(raw)
    elif isinstance(raw, dict):
        # Try common wrapper keys; fall back to normalising the whole dict
        for key in ('data', 'products', 'results', 'items'):
            if key in raw:
                df = pd.json_normalize(raw[key])
                break
        else:
            df = pd.json_normalize(raw)
    else:
        df = pd.DataFrame()
    print(f'  ✓  Loaded  {filename:<45} → {df.shape[0]:>8,} rows × {df.shape[1]} cols')
    return df


def standardise_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercase + strip + replace spaces/hyphens with underscores."""
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(r'[\s\-]+', '_', regex=True)
          .str.replace(r'[^\w]', '', regex=True)
    )
    return df


def quick_report(df: pd.DataFrame, title: str = '') -> None:
    """Print shape, dtypes, missing values and first 3 rows."""
    print(f'\n{'─'*70}')
    if title:
        print(f'  📋  {title}')
    print(f'  Rows : {df.shape[0]:,}   Cols : {df.shape[1]}')
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        print('  Missing values:')
        for col, n in missing.items():
            print(f'    {col:<40} {n:>7,} ({n/len(df)*100:.1f}%)')
    else:
        print('  No missing values.')
    print(f'{'─'*70}')
    display(df.head(3))


def basic_clean(df: pd.DataFrame,
                drop_dup_subset=None,
                drop_all_na_rows: bool = True) -> pd.DataFrame:
    """Apply standard cleaning steps."""
    before = len(df)
    # 1. Standardise column names
    df = standardise_columns(df)
    # 2. Drop fully-empty rows
    if drop_all_na_rows:
        df = df.dropna(how='all')
    # 3. Strip whitespace from string columns
    str_cols = df.select_dtypes(include='object').columns
    df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())
    # 4. Replace empty strings with NaN
    df.replace('', np.nan, inplace=True)
    # 5. Drop exact duplicates
    df = df.drop_duplicates(subset=drop_dup_subset)
    after = len(df)
    removed = before - after
    if removed:
        print(f'  ✂  Removed {removed:,} duplicate / empty rows  ({before:,} → {after:,})')
    return df.reset_index(drop=True)


def save(df: pd.DataFrame, filename: str) -> None:
    """Save cleaned DataFrame to OUTPUT_DIR as CSV."""
    out = OUTPUT_DIR / filename
    df.to_csv(out, index=False)
    print(f'  💾  Saved → {out}  [{df.shape[0]:,} rows × {df.shape[1]} cols]')


print('✅  Helper functions loaded.')

✅  Helper functions loaded.


---
## 📝 Section 1 — Reviews (Combined)
**Source files:** `reviews_0-250`, `reviews_250-500`, `reviews_500-750`, `reviews_750-1250`, `reviews_1250-end`, `productReviews`  
**Output:** `01_reviews_clean.csv`

These files are chunked slices of the same review dataset — we combine them, deduplicate and clean.

In [53]:
print('='*70)
print('SECTION 1: REVIEWS')
print('='*70 + '\n')

review_files = ['reviews_0-250.csv', 'reviews_250-500.csv', 'reviews_500-750.csv',
                'reviews_750-1250.csv', 'reviews_1250-end.csv']

print('Loading review chunks...')
review_chunks = [load_file(fname) for fname in review_files]
review_chunks = [df for df in review_chunks if not df.empty]

if review_chunks:
    print(f'\nCombining {len(review_chunks)} chunks...')
    reviews = pd.concat([standardise_columns(df.copy()) for df in review_chunks], ignore_index=True)
    reviews = basic_clean(reviews)
    
    for col in ['rating', 'helpful_votes', 'total_votes', 'score']:
        if col in reviews.columns:
            reviews[col] = pd.to_numeric(reviews[col], errors='coerce')
    
    print('\nSaving...')
    save(reviews, '01_reviews_clean.csv')
else:
    print('⚠️  No review files found')

SECTION 1: REVIEWS

Loading review chunks...
  ✓  Loaded  reviews_0-250.csv                             →  602,130 rows × 19 cols
  ✓  Loaded  reviews_250-500.csv                           →  206,725 rows × 19 cols
  ✓  Loaded  reviews_500-750.csv                           →  116,262 rows × 19 cols
  ✓  Loaded  reviews_750-1250.csv                          →  119,317 rows × 19 cols
  ✓  Loaded  reviews_1250-end.csv                          →   49,977 rows × 19 cols

Combining 5 chunks...

Saving...
  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\01_reviews_clean.csv  [1,094,411 rows × 19 cols]


In [54]:
# ── Preview column alignment before concat ───────────────────────────────────
print('Column sets in each chunk:')
for i, df in enumerate(review_chunks):
    print(f'  Chunk {i}: {list(df.columns)}')

if not product_reviews.empty:
    print(f'  productReviews: {list(product_reviews.columns)}')

Column sets in each chunk:
  Chunk 0: ['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd']
  Chunk 1: ['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd']
  Chunk 2: ['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd']
  Chunk

In [55]:
# ── Standardise and combine ──────────────────────────────────────────────────
review_chunks_std = [standardise_columns(df.copy()) for df in review_chunks]

reviews = pd.concat(review_chunks_std, ignore_index=True)
print(f'Combined review chunks  → {reviews.shape[0]:,} rows × {reviews.shape[1]} cols')

# Optionally merge productReviews if column structure matches
if not product_reviews.empty:
    pr = standardise_columns(product_reviews.copy())
    common_cols = list(set(reviews.columns) & set(pr.columns))
    print(f'Common columns with productReviews: {common_cols}')
    # ⚠️  Uncomment line below only if structures match:
    # reviews = pd.concat([reviews, pr[common_cols]], ignore_index=True)
    # print(f'After merging productReviews → {reviews.shape[0]:,} rows')

Combined review chunks  → 1,094,411 rows × 19 cols


In [56]:
# ── Clean ─────────────────────────────────────────────────────────────────────
# ⚠️  Adjust 'review_id' to whatever unique review identifier your data uses
reviews_clean = basic_clean(reviews, drop_dup_subset=None)

# ── Type fixes ────────────────────────────────────────────────────────────────
# Numeric columns — adjust names to match your data
num_candidates = ['rating', 'helpfulness', 'helpful_votes', 'total_votes',
                  'stars', 'score', 'review_id']
for col in num_candidates:
    if col in reviews_clean.columns:
        reviews_clean[col] = pd.to_numeric(reviews_clean[col], errors='coerce')

# Date columns — adjust names to match your data
date_candidates = ['date', 'review_date', 'submission_time', 'created_at']
for col in date_candidates:
    if col in reviews_clean.columns:
        reviews_clean[col] = pd.to_datetime(reviews_clean[col], errors='coerce')

quick_report(reviews_clean, 'Reviews — cleaned')


──────────────────────────────────────────────────────────────────────
  📋  Reviews — cleaned
  Rows : 1,094,411   Cols : 19
  Missing values:
    author_id                                193,355 (17.7%)
    is_recommended                           167,988 (15.3%)
    helpfulness                              561,592 (51.3%)
    review_text                                1,444 (0.1%)
    review_title                             310,654 (28.4%)
    skin_tone                                170,539 (15.6%)
    eye_color                                209,628 (19.2%)
    skin_type                                111,557 (10.2%)
    hair_color                               226,768 (20.7%)
──────────────────────────────────────────────────────────────────────


,unnamed_0,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,0,NaN,5,1.00,1.00,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Balm & Make-Up Melt“ to double cl...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.00
1,1,NaN,1,0.00,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the reviews and the hype. Unfortunately...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,24.00
2,2,NaN,5,1.00,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited to get into bed and apply this...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,24.00


In [57]:
save(reviews_clean, '01_reviews_clean.csv')

  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\01_reviews_clean.csv  [1,094,411 rows × 19 cols]


---
## 💆 Section 2 — Skincare Products (Combined)
**Source files:** `skincare_df`, `moisturizers`, `treatments`, `eyecare`, `wellness`  
**Output:** `02_skincare_products_clean.csv`

These files share the same product structure but cover different sub-categories.

In [58]:
print('─── Loading skincare product files ────────────────────────────────────')

skincare_sources = {
    'skincare':    load_file('skincare_df.csv'),
    'moisturizer': load_file('moisturizers.csv'),
    'treatment':   load_file('treatments.csv'),
    'eyecare':     load_file('eyecare.csv'),
    'wellness':    load_file('wellness.csv'),
}

─── Loading skincare product files ────────────────────────────────────
  ✓  Loaded  skincare_df.csv                               →    1,689 rows × 47 cols
  ✓  Loaded  moisturizers.csv                              →      835 rows × 10 cols
  ✓  Loaded  treatments.csv                                →      577 rows × 10 cols
  ✓  Loaded  eyecare.csv                                   →      221 rows × 10 cols
  ✓  Loaded  wellness.csv                                  →      152 rows × 10 cols


In [59]:
# ── Preview columns per file ─────────────────────────────────────────────────
print('Column sets per skincare file:')
for label, df in skincare_sources.items():
    if not df.empty:
        print(f'  {label:<15} ({df.shape[0]:>5,} rows): {list(df.columns)}')

Column sets per skincare file:
  skincare        (1,689 rows): ['Unnamed: 0', 'brand', 'name', 'price', 'n_of_reviews', 'n_of_loves', 'review_score', 'size', 'clean_product', 'category_Anti-Aging', 'category_BB_&_CC_Cream', 'category_Bath_&_Shower', 'category_Beauty_Supplements', 'category_Blemish_&_Acne_Treatments', 'category_Blotting_Papers', 'category_Body_Lotions_&_Body_Oils', 'category_Cellulite_&_Stretch_Marks', 'category_Decollete_&_Neck_Creams', 'category_Exfoliators', 'category_Eye_Creams_&_Treatments', 'category_Eye_Masks', 'category_Face_Masks', 'category_Face_Oils', 'category_Face_Primer', 'category_Face_Serums', 'category_Face_Sunscreen', 'category_Face_Wash_&_Cleansers', 'category_Facial_Peels', 'category_Foundation', 'category_Hair_Oil', 'category_Highlighter', 'category_Holistic_Wellness', 'category_Mini_Size', 'category_Mists_&_Essences', 'category_Moisturizer_&_Treatments', 'category_Moisturizers', 'category_Night_Creams', 'category_Setting_Spray_&_Powder', 'category_

In [60]:
# ── Tag each source with its sub-category and combine ────────────────────────
skincare_parts = []
for label, df in skincare_sources.items():
    if df.empty:
        continue
    df_std = standardise_columns(df.copy())
    df_std['sub_category'] = label       # tag the source
    skincare_parts.append(df_std)

skincare = pd.concat(skincare_parts, ignore_index=True)
print(f'Combined skincare → {skincare.shape[0]:,} rows × {skincare.shape[1]} cols')

Combined skincare → 3,474 rows × 49 cols


In [61]:
# ── Clean ─────────────────────────────────────────────────────────────────────
skincare_clean = basic_clean(skincare)

# Numeric columns — adjust as needed
num_candidates = ['price', 'rating', 'number_of_reviews', 'loves_count',
                  'review_count', 'score', 'rank']
for col in num_candidates:
    if col in skincare_clean.columns:
        # Remove currency symbols and commas before converting
        if skincare_clean[col].dtype == object:
            skincare_clean[col] = skincare_clean[col].str.replace(r'[\$,£€]', '', regex=True)
        skincare_clean[col] = pd.to_numeric(skincare_clean[col], errors='coerce')

quick_report(skincare_clean, 'Skincare Products — cleaned')


──────────────────────────────────────────────────────────────────────
  📋  Skincare Products — cleaned
  Rows : 3,474   Cols : 49
  Missing values:
    price                                      2,064 (59.4%)
    size                                       1,942 (55.9%)
    category_anti_aging                        1,785 (51.4%)
    category_bb__cc_cream                      1,785 (51.4%)
    category_bath__shower                      1,785 (51.4%)
    category_beauty_supplements                1,785 (51.4%)
    category_blemish__acne_treatments          1,785 (51.4%)
    category_blotting_papers                   1,785 (51.4%)
    category_body_lotions__body_oils           1,785 (51.4%)
    category_cellulite__stretch_marks          1,785 (51.4%)
    category_decollete__neck_creams            1,785 (51.4%)
    category_exfoliators                       1,785 (51.4%)
    category_eye_creams__treatments            1,785 (51.4%)
    category_eye_masks                         1,785 (51.

,unnamed_0,brand,name,price,n_of_reviews,n_of_loves,review_score,size,clean_product,category_anti_aging,category_bb__cc_cream,category_bath__shower,category_beauty_supplements,category_blemish__acne_treatments,category_blotting_papers,category_body_lotions__body_oils,category_cellulite__stretch_marks,category_decollete__neck_creams,category_exfoliators,category_eye_creams__treatments,category_eye_masks,category_face_masks,category_face_oils,category_face_primer,category_face_serums,category_face_sunscreen,category_face_wash__cleansers,category_facial_peels,category_foundation,category_hair_oil,category_highlighter,category_holistic_wellness,category_mini_size,category_mists__essences,category_moisturizer__treatments,category_moisturizers,category_night_creams,category_setting_spray__powder,category_sheet_masks,category_skincare,category_tinted_moisturizer,category_toners,category_tools,category_value__gift_sets,reviews_to_loves_ratio,return_on_reviews,price_per_ounce,sub_category,category
0,0,Drunk Elephant,Protini Polypeptide Moisturizer,NaN,1000,136008,4.21,NaN,1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.74,0.42,40.24,skincare,NaN
1,1,La Mer,Crreme de la Mer,NaN,493,61648,4.10,NaN,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.80,0.83,175.00,skincare,NaN
2,2,IT Cosmetics,CC+ Cream with SPF 50+,NaN,2000,188389,4.04,NaN,0,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.06,0.20,36.11,skincare,NaN


In [62]:
print('Sub-category breakdown:')
print(skincare_clean['sub_category'].value_counts().to_string())
save(skincare_clean, '02_skincare_products_clean.csv')

Sub-category breakdown:
sub_category
skincare       1689
moisturizer     835
treatment       577
eyecare         221
wellness        152
  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\02_skincare_products_clean.csv  [3,474 rows × 49 cols]


---
## 🔗 Section 3 — Product URLs (Combined)
**Source files:** `product_urls`, `eyecare_url`, `treatment_urls`, `wellness_urls`  
**Output:** `03_product_urls_clean.csv`

In [64]:
print('─── Loading URL files ──────────────────────────────────────────────────')

url_sources = {
    'general':   load_file('product_urls.csv'),
    'eyecare':   load_file('eyecare_url.csv'),
    'treatment': load_file('treatment_urls.csv'),
    'wellness':  load_file('wellness_urls.csv'),
}

─── Loading URL files ──────────────────────────────────────────────────
  ✓  Loaded  product_urls.csv                              →      836 rows × 2 cols
  ✓  Loaded  eyecare_url.csv                               →      224 rows × 2 cols
  ✓  Loaded  treatment_urls.csv                            →      576 rows × 2 cols
  ✓  Loaded  wellness_urls.csv                             →      158 rows × 2 cols


In [65]:
url_parts = []
for label, df in url_sources.items():
    if df.empty:
        continue
    df_std = standardise_columns(df.copy())
    df_std['url_category'] = label
    url_parts.append(df_std)

urls = pd.concat(url_parts, ignore_index=True)
print(f'Combined URLs → {urls.shape[0]:,} rows × {urls.shape[1]} cols')

Combined URLs → 1,794 rows × 6 cols


In [66]:
urls_clean = basic_clean(urls)

# Detect and validate URL columns
url_cols = [c for c in urls_clean.columns if 'url' in c.lower() or 'link' in c.lower()]
print(f'URL columns detected: {url_cols}')

for col in url_cols:
    # Flag malformed URLs (basic check)
    mask = urls_clean[col].notna() & ~urls_clean[col].str.startswith('http', na=False)
    if mask.any():
        print(f'  ⚠  {mask.sum()} entries in [{col}] don\'t start with "http"')

# Drop exact duplicate URLs
if url_cols:
    urls_clean = urls_clean.drop_duplicates(subset=url_cols)
    print(f'After URL dedup → {len(urls_clean):,} rows')

quick_report(urls_clean, 'Product URLs — cleaned')
save(urls_clean, '03_product_urls_clean.csv')

URL columns detected: ['url_category']
  ⚠  1794 entries in [url_category] don't start with "http"
After URL dedup → 4 rows

──────────────────────────────────────────────────────────────────────
  📋  Product URLs — cleaned
  Rows : 4   Cols : 6
  Missing values:
    httpswwwsephoracomproductprotini_tm_polypeptide_cream_p427421icid2products20gridp427421product       3 (75.0%)
    httpswwwsephoracomproductbanana_bright_eye_creme_p426339icid2products20gridp426339product       3 (75.0%)
    httpswwwsephoracomproductdouble_serum_complete_age_control_concentrate_p421235icid2products20gridp421235product       3 (75.0%)
    httpswwwsephoracomproductsilk_pillowcase_standard_queen_p402944icid2products20gridp402944product       3 (75.0%)
──────────────────────────────────────────────────────────────────────


,0,httpswwwsephoracomproductprotini_tm_polypeptide_cream_p427421icid2products20gridp427421product,url_category,httpswwwsephoracomproductbanana_bright_eye_creme_p426339icid2products20gridp426339product,httpswwwsephoracomproductdouble_serum_complete_age_control_concentrate_p421235icid2products20gridp421235product,httpswwwsephoracomproductsilk_pillowcase_standard_queen_p402944icid2products20gridp402944product
0,1,https://www.sephora.com/product/creme-de-la-mer-moisturizing-cream-P416341?i...,general,NaN,NaN,NaN
836,1,NaN,eyecare,https://www.sephora.com/product/c-tango-multivitamin-eye-cream-P429515?icid2...,NaN,NaN
1060,1,NaN,treatment,NaN,https://www.sephora.com/product/c-firma-day-serum-P400259?icid2=products%20g...,NaN


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\03_product_urls_clean.csv  [4 rows × 6 cols]


---
## 📊 Section 4 — Popularity Rankings (Combined)
**Source files:** `most_expensive`, `most_popular_by_loves`, `most_popular_by_n_reviews`, `most_popular_by_score`  
**Output:** `04_popularity_rankings_clean.csv`

In [67]:
print('─── Loading popularity/ranking files ──────────────────────────────────')

popularity_sources = {
    'most_expensive':        load_file('most_expensive.csv'),
    'most_popular_by_loves': load_file('most_popular_by_loves.csv'),
    'most_popular_by_n_reviews': load_file('most_popular_by_n_reviews.csv'),
    'most_popular_by_score': load_file('most_popular_by_score.csv'),
}

─── Loading popularity/ranking files ──────────────────────────────────
  ✓  Loaded  most_expensive.csv                            →      100 rows × 4 cols
  ✓  Loaded  most_popular_by_loves.csv                     →      100 rows × 3 cols
  ✓  Loaded  most_popular_by_n_reviews.csv                 →      100 rows × 3 cols
  ✓  Loaded  most_popular_by_score.csv                     →      100 rows × 3 cols


In [68]:
print('Column comparison across popularity files:')
for label, df in popularity_sources.items():
    if not df.empty:
        print(f'  {label:<35} ({df.shape[0]:>4,} rows): {list(df.columns)}')

Column comparison across popularity files:
  most_expensive                      ( 100 rows): ['Unnamed: 0', 'brand', 'name', 'price']
  most_popular_by_loves               ( 100 rows): ['Unnamed: 0', 'brand', 'SUM(n_of_loves)']
  most_popular_by_n_reviews           ( 100 rows): ['Unnamed: 0', 'brand', 'SUM(n_of_reviews)']
  most_popular_by_score               ( 100 rows): ['Unnamed: 0', 'brand', 'SUM(review_score)']


In [69]:
pop_parts = []
for label, df in popularity_sources.items():
    if df.empty:
        continue
    df_std = standardise_columns(df.copy())
    df_std['ranking_type'] = label
    pop_parts.append(df_std)

popularity = pd.concat(pop_parts, ignore_index=True)
popularity_clean = basic_clean(popularity)

# Make numeric columns numeric
num_candidates = ['price', 'rating', 'loves', 'loves_count',
                  'number_of_reviews', 'review_count', 'score', 'rank']
for col in num_candidates:
    if col in popularity_clean.columns:
        if popularity_clean[col].dtype == object:
            popularity_clean[col] = popularity_clean[col].str.replace(r'[\$,£€]', '', regex=True)
        popularity_clean[col] = pd.to_numeric(popularity_clean[col], errors='coerce')

quick_report(popularity_clean, 'Popularity Rankings — cleaned')
print('\nRanking type counts:')
print(popularity_clean['ranking_type'].value_counts().to_string())
save(popularity_clean, '04_popularity_rankings_clean.csv')


──────────────────────────────────────────────────────────────────────
  📋  Popularity Rankings — cleaned
  Rows : 400   Cols : 8
  Missing values:
    name                                         300 (75.0%)
    price                                        300 (75.0%)
    sumn_of_loves                                300 (75.0%)
    sumn_of_reviews                              300 (75.0%)
    sumreview_score                              300 (75.0%)
──────────────────────────────────────────────────────────────────────


,unnamed_0,brand,name,price,ranking_type,sumn_of_loves,sumn_of_reviews,sumreview_score
0,0,Perricone MD,Neuropeptide Smoothing Facial Conformer,495.00,most_expensive,NaN,NaN,NaN
1,1,Guerlain,Orchid√©e Imp√©riale The Cream,460.00,most_expensive,NaN,NaN,NaN
2,2,SK-II,Ultimate Revival Cream,385.00,most_expensive,NaN,NaN,NaN



Ranking type counts:
ranking_type
most_expensive               100
most_popular_by_loves        100
most_popular_by_n_reviews    100
most_popular_by_score        100
  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\04_popularity_rankings_clean.csv  [400 rows × 8 cols]


---
## 📦 Section 5 — Product Info
**Source:** `product_info` (7.7 MB)  
**Output:** `05_product_info_clean.csv`

In [70]:
print('─── Loading product_info ───────────────────────────────────────────────')
product_info = load_file('product_info.csv')
quick_report(standardise_columns(product_info.copy().head(5)), 'product_info — preview')

─── Loading product_info ───────────────────────────────────────────────
  ✓  Loaded  product_info.csv                              →    8,494 rows × 27 cols

──────────────────────────────────────────────────────────────────────
  📋  product_info — preview
  Rows : 5   Cols : 27
  Missing values:
    size                                           1 (20.0%)
    variation_type                                 1 (20.0%)
    variation_value                                1 (20.0%)
    variation_desc                                 5 (100.0%)
    value_price_usd                                5 (100.0%)
    sale_price_usd                                 5 (100.0%)
    child_max_price                                1 (20.0%)
    child_min_price                                1 (20.0%)
──────────────────────────────────────────────────────────────────────


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,variation_desc,ingredients,price_usd,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.64,11.00,NaN,NaN,NaN,NaN,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance...",35.00,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scent', 'Woody & Earthy Scent', 'F...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.15,13.00,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Ethylhexyl Methoxycinn...",195.00,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Warm &Spicy Scent']",Fragrance,Women,Perfume,2,85.00,30.00
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.25,16.00,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Ethylhexyl...",195.00,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Woody & Earthy Scent']",Fragrance,Women,Perfume,2,75.00,30.00


In [71]:
pi_clean = basic_clean(product_info)

# Price / rating cleaning
price_cols = [c for c in pi_clean.columns if 'price' in c.lower()]
for col in price_cols:
    pi_clean[col] = pi_clean[col].astype(str).str.replace(r'[\$,£€]', '', regex=True)
    pi_clean[col] = pd.to_numeric(pi_clean[col], errors='coerce')

rating_cols = [c for c in pi_clean.columns if 'rating' in c.lower() or 'score' in c.lower()]
for col in rating_cols:
    pi_clean[col] = pd.to_numeric(pi_clean[col], errors='coerce')
    # Clamp ratings to sensible range (e.g. 0-5)
    pi_clean[col] = pi_clean[col].clip(lower=0, upper=5)

quick_report(pi_clean, 'Product Info — cleaned')
save(pi_clean, '05_product_info_clean.csv')


──────────────────────────────────────────────────────────────────────
  📋  Product Info — cleaned
  Rows : 8,494   Cols : 27
  Missing values:
    rating                                       278 (3.3%)
    reviews                                      278 (3.3%)
    size                                       1,631 (19.2%)
    variation_type                             1,444 (17.0%)
    variation_value                            1,598 (18.8%)
    variation_desc                             7,244 (85.3%)
    ingredients                                  945 (11.1%)
    value_price_usd                            8,043 (94.7%)
    sale_price_usd                             8,224 (96.8%)
    highlights                                 2,207 (26.0%)
    secondary_category                             8 (0.1%)
    tertiary_category                            990 (11.7%)
    child_max_price                            5,740 (67.6%)
    child_min_price                            5,740 (67.6%)
────

,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,variation_desc,ingredients,price_usd,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.64,11.00,NaN,NaN,NaN,NaN,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance...",35.00,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scent', 'Woody & Earthy Scent', 'F...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.15,13.00,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Ethylhexyl Methoxycinn...",195.00,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Warm &Spicy Scent']",Fragrance,Women,Perfume,2,85.00,30.00
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.25,16.00,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Ethylhexyl...",195.00,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Woody & Earthy Scent']",Fragrance,Women,Perfume,2,75.00,30.00


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\05_product_info_clean.csv  [8,494 rows × 27 cols]


---
## 📋 Section 6 — Product List
**Source:** `productlist` (203 KB)  
**Output:** `06_productlist_clean.csv`

In [72]:
print('─── Loading productlist ────────────────────────────────────────────────')
productlist = load_file('productlist.csv')
pl_clean = basic_clean(productlist)

for col in pl_clean.select_dtypes(include='object').columns:
    if 'price' in col.lower():
        pl_clean[col] = pl_clean[col].str.replace(r'[\$,£€]', '', regex=True)
        pl_clean[col] = pd.to_numeric(pl_clean[col], errors='coerce')

quick_report(pl_clean, 'Product List — cleaned')
save(pl_clean, '06_productlist_clean.csv')

─── Loading productlist ────────────────────────────────────────────────
  ✓  Loaded  productlist.csv                               →      237 rows × 7 cols

──────────────────────────────────────────────────────────────────────
  📋  Product List — cleaned
  Rows : 237   Cols : 7
  No missing values.
──────────────────────────────────────────────────────────────────────


,unnamed_0,product_id,product_name,product_brand,price,product_description,product_type
0,0,6562638659653,VITALIFT-A,Dr. Different,42.00,This night-time skin treatment is ideal for those looking to improve the app...,Other/Spot Treatments
1,1,6562639675461,VITALIFT-A Forte,Dr. Different,52.00,Those that need an extra boost to smooth out fine lines and wrinkles and rea...,Other/Spot Treatments
2,2,6562640429125,VITALIFT-A Eye & Neck,Dr. Different,40.00,For those looking to target fine lines and wrinkles specifically around the ...,Eye Treatment


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\06_productlist_clean.csv  [237 rows × 7 cols]


---
## 🛍️ Section 7 — Sephora All Products
**Source:** `Sephora_all_423` (8 MB)  
**Output:** `07_sephora_clean.csv`

In [73]:
print('─── Loading Sephora_all_423 ────────────────────────────────────────────')
sephora = load_file('Sephora_all_423.csv')
sephora_clean = basic_clean(sephora)

print(f'\nColumns: {list(sephora_clean.columns)}')

# Price cleaning
for col in sephora_clean.columns:
    if 'price' in col.lower():
        sephora_clean[col] = sephora_clean[col].astype(str).str.replace(r'[\$,£€]', '', regex=True)
        sephora_clean[col] = pd.to_numeric(sephora_clean[col], errors='coerce')

# Rating cleaning
for col in sephora_clean.columns:
    if 'rating' in col.lower():
        sephora_clean[col] = pd.to_numeric(sephora_clean[col], errors='coerce').clip(0, 5)

# Standardise boolean-like columns
bool_candidates = ['is_exclusive', 'is_new', 'is_bestseller', 'is_sale',
                   'limited_edition', 'online_only', 'out_of_stock']
for col in bool_candidates:
    if col in sephora_clean.columns:
        sephora_clean[col] = sephora_clean[col].map(
            {1: True, 0: False, '1': True, '0': False,
             'true': True, 'false': False, 'yes': True, 'no': False}
        )

quick_report(sephora_clean, 'Sephora Products — cleaned')
save(sephora_clean, '07_sephora_clean.csv')

─── Loading Sephora_all_423 ────────────────────────────────────────────
  ✓  Loaded  Sephora_all_423.csv                           →    2,179 rows × 20 cols

Columns: ['cosmetic_link', 'brand_name', 'cosmetic_name', 'num_customer', 'price', 'ingredients', 'about', 'reviews', 'recommended', 'what_it_is', 'skin_type', 'skincare_concerns', 'formulation', 'benefits', 'highlighted_ingredients', 'ingredient_callouts', 'what_else_you_need_to_know', 'clinical_results', 'clean_ingredients', 'new_ingredients']

──────────────────────────────────────────────────────────────────────
  📋  Sephora Products — cleaned
  Rows : 2,179   Cols : 20
  Missing values:
    num_customer                                  10 (0.5%)
    price                                        381 (17.5%)
    about                                        244 (11.2%)
    recommended                                  866 (39.7%)
    what_it_is                                   264 (12.1%)
    skin_type                           

,cosmetic_link,brand_name,cosmetic_name,num_customer,price,ingredients,about,reviews,recommended,what_it_is,skin_type,skincare_concerns,formulation,benefits,highlighted_ingredients,ingredient_callouts,what_else_you_need_to_know,clinical_results,clean_ingredients,new_ingredients
0,https://www.sephora.com/product/summer-fridays-lip-butter-balm-P455936?skuId...,Summer Fridays,Lip Butter Balm for Hydration & Shine,6.7K,24.00,"-Shea and Murumuru Seed Butters: Natural moisturizers that soothe, relieve, ...",What it is: A silky vegan balm that hydrates and soothes dry lips in seconds...,4.40,86%,A silky vegan balm that hydrates and soothes dry lips in seconds.,NaN,Dryness and Dullness,NaN,NaN,NaN,"This product is vegan, gluten-free, cruelty-free, and comes in recyclable pa...",This formula delivers soothing moisture to parched lips in seconds. Butter u...,"In an independent clinical study, with 39 participants aged 20 - 55:",NaN,"Phytosteryl/Behenyl Dimer Dilinoleate, Diisostearyl Malate, Hydrogenated Pol..."
1,https://www.sephora.com/product/glow-recipe-watermelon-glow-pha-bha-pore-tig...,Glow Recipe,Watermelon Glow PHA + BHA Pore-Tight Toner,6.1K,NaN,"-Watermelon Extract: Hydrates, soothes, and delivers essential vitamins and ...","What it is: A bestselling, gentle PHA- and BHA-infused watermelon toner that...",4.30,84%,"A bestselling, gentle PHA- and BHA-infused watermelon toner that hydrates, g...","Normal, Dry, Combination, and Oily","Pores, Dryness, and Dullness",Lightweight Liquid,NaN,NaN,"This product is vegan, cruelty-free, and comes in recyclable packaging.","Suitable for all skin types, this bouncy, alcohol-free toner is made with PH...",Based on instrumental testing on 31 women when used as directed After 2 week...,NaN,"Opuntia Ficus-Indica Stem Extract, Citrullus Lanatus (Watermelon) Fruit Extr..."
2,https://www.sephora.com/product/touchland-power-mist-hydrating-hand-sanitize...,Touchland,Power Mist Hydrating Hand Sanitizer,845,10.00,-70% Alcohol: Effective against most common germs.\n-Aloe Vera: Moisturizes ...,"What it is: An award-winning hand sanitizer that does it all: sanitizes, hyd...",4.20,84%,"An award-winning hand sanitizer that does it all: sanitizes, hydrates, and s...",NaN,NaN,NaN,NaN,- 70% Alcohol: Effective against most common germs.\n- Aloe Vera: Moisturize...,NaN,This revolutionary sanitizer has turned hand hygiene into a ritual of skinca...,NaN,NaN,"Alcohol, Deionized/Demineralized Water, Aloe Barbadensis Leaf Juice, Leucono..."


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\07_sephora_clean.csv  [2,179 rows × 20 cols]


---
## 🏷️ Section 8 — Brands
**Source:** `brands_w_m_products` (2 KB)  
**Output:** `08_brands_clean.csv`

In [76]:
brands = load_file("brands_w_m_products.csv")
brands_clean = basic_clean(brands)

# Capitalise brand/name columns safely
name_cols = [
    c for c in brands_clean.columns
    if "brand" in c.lower() or "name" in c.lower()
]

for col in name_cols:
    brands_clean[col] = (
        brands_clean[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

quick_report(brands_clean, "Brands - cleaned")
save(brands_clean, "08_brands_clean.csv")

  ✓  Loaded  brands_w_m_products.csv                       →      100 rows × 3 cols

──────────────────────────────────────────────────────────────────────
  📋  Brands - cleaned
  Rows : 100   Cols : 3
  No missing values.
──────────────────────────────────────────────────────────────────────


,unnamed_0,brand,countbrand
0,0,Clinique,64
1,1,Murad,54
2,2,Perricone Md,52


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\08_brands_clean.csv  [100 rows × 3 cols]


---
## 💄 Section 9 — Makeup Data (JSON)
**Source:** `makeup_data.json` (2.1 MB)  
**Output:** `09_makeup_clean.csv`

This is the only JSON file in the dataset — we normalise it and clean.

In [77]:
print('─── Loading makeup_data.json ───────────────────────────────────────────')
makeup = load_json('makeup_data.json')

if not makeup.empty:
    print(f'Columns: {list(makeup.columns)}')
    display(makeup.head(3))

─── Loading makeup_data.json ───────────────────────────────────────────
  ✓  Loaded  makeup_data.json                              →      931 rows × 19 cols
Columns: ['id', 'brand', 'name', 'price', 'price_sign', 'currency', 'image_link', 'product_link', 'website_link', 'description', 'rating', 'category', 'product_type', 'tag_list', 'created_at', 'updated_at', 'product_api_url', 'api_featured_image', 'product_colors']


,id,brand,name,price,price_sign,currency,image_link,product_link,website_link,description,rating,category,product_type,tag_list,created_at,updated_at,product_api_url,api_featured_image,product_colors
0,1048,colourpop,Lippie Pencil,5.0,$,CAD,https://cdn.shopify.com/s/files/1/1338/0845/collections/lippie-pencil_grande...,https://colourpop.com/collections/lippie-pencil,https://colourpop.com,Lippie Pencil A long-wearing and high-intensity lip pencil that glides on ea...,NaN,pencil,lip_liner,"[cruelty free, Vegan]",2018-07-08T23:45:08.056Z,2018-07-09T00:53:23.301Z,http://makeup-api.herokuapp.com/api/v1/products/1048.json,//s3.amazonaws.com/donovanbailey/products/api_featured_images/000/001/048/or...,"[{'hex_value': '#B28378', 'colour_name': 'BFF Pencil'}, {'hex_value': '#A36B..."
1,1047,colourpop,Blotted Lip,5.5,$,CAD,https://cdn.shopify.com/s/files/1/1338/0845/products/brain-freeze_a_800x1200...,https://colourpop.com/collections/lippie-stix?filter=blotted-lip,https://colourpop.com,Blotted Lip Sheer matte lipstick that creates the perfect popsicle pout! For...,NaN,lipstick,lipstick,"[cruelty free, Vegan]",2018-07-08T22:01:20.178Z,2018-07-09T00:53:23.287Z,http://makeup-api.herokuapp.com/api/v1/products/1047.json,//s3.amazonaws.com/donovanbailey/products/api_featured_images/000/001/047/or...,"[{'hex_value': '#b72227', 'colour_name': 'Bee's Knees'}, {'hex_value': '#BB6..."
2,1046,colourpop,Lippie Stix,5.5,$,CAD,https://cdn.shopify.com/s/files/1/1338/0845/collections/blottedlip-lippie-st...,https://colourpop.com/collections/lippie-stix,https://colourpop.com,"Lippie Stix Formula contains Vitamin E, Mango, Avocado, and Shea butter for ...",NaN,lipstick,lipstick,"[cruelty free, Vegan]",2018-07-08T21:47:49.858Z,2018-07-09T00:53:23.274Z,http://makeup-api.herokuapp.com/api/v1/products/1046.json,//s3.amazonaws.com/donovanbailey/products/api_featured_images/000/001/046/or...,"[{'hex_value': '#F2DEC3', 'colour_name': 'Fair 05'}, {'hex_value': '#793C36'..."


In [78]:
if not makeup.empty:
    makeup_clean = basic_clean(makeup)

    # Price / rating numeric
    for col in makeup_clean.columns:
        if 'price' in col.lower():
            makeup_clean[col] = makeup_clean[col].astype(str).str.replace(r'[\$,£€]', '', regex=True)
            makeup_clean[col] = pd.to_numeric(makeup_clean[col], errors='coerce')
        if 'rating' in col.lower():
            makeup_clean[col] = pd.to_numeric(makeup_clean[col], errors='coerce').clip(0, 5)

    # Flatten any nested list/dict columns
    for col in makeup_clean.columns:
        if makeup_clean[col].apply(lambda x: isinstance(x, (list, dict))).any():
            print(f'  ℹ  Column [{col}] contains nested objects — converting to string')
            makeup_clean[col] = makeup_clean[col].apply(
                lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x
            )

    quick_report(makeup_clean, 'Makeup Products — cleaned')
    save(makeup_clean, '09_makeup_clean.csv')
else:
    print('  ⚠  Makeup data is empty — skipping')


──────────────────────────────────────────────────────────────────────
  📋  Makeup Products — cleaned
  Rows : 931   Cols : 19
  Missing values:
    brand                                         12 (1.3%)
    price                                         14 (1.5%)
    price_sign                                   931 (100.0%)
    currency                                     563 (60.5%)
    description                                   25 (2.7%)
    rating                                       591 (63.5%)
    category                                     424 (45.5%)
    tag_list                                     931 (100.0%)
    product_colors                               931 (100.0%)
──────────────────────────────────────────────────────────────────────


,id,brand,name,price,price_sign,currency,image_link,product_link,website_link,description,rating,category,product_type,tag_list,created_at,updated_at,product_api_url,api_featured_image,product_colors
0,1048,colourpop,Lippie Pencil,5.00,NaN,CAD,https://cdn.shopify.com/s/files/1/1338/0845/collections/lippie-pencil_grande...,https://colourpop.com/collections/lippie-pencil,https://colourpop.com,Lippie Pencil A long-wearing and high-intensity lip pencil that glides on ea...,NaN,pencil,lip_liner,NaN,2018-07-08T23:45:08.056Z,2018-07-09T00:53:23.301Z,http://makeup-api.herokuapp.com/api/v1/products/1048.json,//s3.amazonaws.com/donovanbailey/products/api_featured_images/000/001/048/or...,NaN
1,1047,colourpop,Blotted Lip,5.50,NaN,CAD,https://cdn.shopify.com/s/files/1/1338/0845/products/brain-freeze_a_800x1200...,https://colourpop.com/collections/lippie-stix?filter=blotted-lip,https://colourpop.com,Blotted Lip Sheer matte lipstick that creates the perfect popsicle pout! For...,NaN,lipstick,lipstick,NaN,2018-07-08T22:01:20.178Z,2018-07-09T00:53:23.287Z,http://makeup-api.herokuapp.com/api/v1/products/1047.json,//s3.amazonaws.com/donovanbailey/products/api_featured_images/000/001/047/or...,NaN
2,1046,colourpop,Lippie Stix,5.50,NaN,CAD,https://cdn.shopify.com/s/files/1/1338/0845/collections/blottedlip-lippie-st...,https://colourpop.com/collections/lippie-stix,https://colourpop.com,"Lippie Stix Formula contains Vitamin E, Mango, Avocado, and Shea butter for ...",NaN,lipstick,lipstick,NaN,2018-07-08T21:47:49.858Z,2018-07-09T00:53:23.274Z,http://makeup-api.herokuapp.com/api/v1/products/1046.json,//s3.amazonaws.com/donovanbailey/products/api_featured_images/000/001/046/or...,NaN


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\09_makeup_clean.csv  [931 rows × 19 cols]


---
## 🧪 Section 10 — Cosmetic Ingredients
**Source:** `binary_cosmetic_ingredient` (4.2 MB)  
**Output:** `10_ingredients_clean.csv`

This is a binary presence/absence matrix of ingredients across products — we clean column names, deduplicate, and verify binary encoding.

In [79]:
print('─── Loading binary_cosmetic_ingredient ────────────────────────────────')
ingredients = load_file('binary_cosmetic_ingredient.csv')
print(f'Shape: {ingredients.shape}')
print(f'Columns (first 15): {list(ingredients.columns[:15])}')

─── Loading binary_cosmetic_ingredient ────────────────────────────────
  ✓  Loaded  binary_cosmetic_ingredient.csv                →   66,417 rows × 2 cols
Shape: (66417, 2)
Columns (first 15): ['cosmetic', 'ingredient']


In [80]:
ingredients_clean = basic_clean(ingredients)

# Identify binary ingredient columns (should contain only 0/1)
# First columns are likely product metadata (name, brand, etc.)
print('Unique values in first 5 columns:')
for col in ingredients_clean.columns[:5]:
    print(f'  {col}: {ingredients_clean[col].unique()[:10]}')

# Find ingredient columns (binary 0/1)
binary_cols = []
for col in ingredients_clean.columns:
    unique_vals = set(ingredients_clean[col].dropna().unique())
    if unique_vals.issubset({0, 1, 0.0, 1.0, '0', '1'}):
        binary_cols.append(col)

print(f'\n  Binary ingredient columns detected: {len(binary_cols)}')

# Convert binary columns to integer
for col in binary_cols:
    ingredients_clean[col] = pd.to_numeric(ingredients_clean[col], errors='coerce').fillna(0).astype(int)

# Summary: how many ingredients present per product
if binary_cols:
    ingredients_clean['total_ingredients'] = ingredients_clean[binary_cols].sum(axis=1)
    print(f'  Ingredient count per product — min: {ingredients_clean["total_ingredients"].min()}, '
          f'max: {ingredients_clean["total_ingredients"].max()}, '
          f'mean: {ingredients_clean["total_ingredients"].mean():.1f}')

quick_report(ingredients_clean, 'Ingredients Binary Matrix — cleaned')
save(ingredients_clean, '10_ingredients_clean.csv')

Unique values in first 5 columns:
  cosmetic: ['Lip Butter Balm for Hydration & Shine'
 'Watermelon Glow PHA + BHA Pore-Tight Toner'
 'Power Mist Hydrating Hand Sanitizer'
 'Hyaluronic Acid 2% + B5 Hydrating Serum'
 'Lip Sleeping Mask Intense Hydration with Vitamin C'
 'Watermelon Glow Niacinamide Hue Drops Sun Glow Serum'
 'B-Hydra™ Intensive Hydration Serum with Hyaluronic Acid'
 'Glycolic Acid 7% Exfoliating Toner' 'Lip Glowy Balm'
 'D-Bronzi™ Bronzing Drops with Peptides']
  ingredient: ['Phytosteryl/Behenyl Dimer Dilinoleate' 'Diisostearyl Malate'
 'Hydrogenated Polyisobutene' 'Polybutene'
 'Hydrogenated Dimer Dilinoleyl PEG-41/Poly(1,2-Butanediol)-18 Dimethyl Ether'
 'Butyrospermum Parkii Butter' 'Cera Microcristallina' 'Octyldodecanol'
 'Synthetic Wax' 'Disteardimonium Hectorite']

  Binary ingredient columns detected: 0

──────────────────────────────────────────────────────────────────────
  📋  Ingredients Binary Matrix — cleaned
  Rows : 66,417   Cols : 2
  No missing values.

,cosmetic,ingredient
0,Lip Butter Balm for Hydration & Shine,Phytosteryl/Behenyl Dimer Dilinoleate
1,Lip Butter Balm for Hydration & Shine,Diisostearyl Malate
2,Lip Butter Balm for Hydration & Shine,Hydrogenated Polyisobutene


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\10_ingredients_clean.csv  [66,417 rows × 2 cols]


---
## 🔬 Section 11 — Paula's Choice — SUM LIST
**Source:** `Paula_SUM_LIST` (5 MB)  
**Output:** `11_paula_sumlist_clean.csv`

In [81]:
print('─── Loading Paula_SUM_LIST ─────────────────────────────────────────────')
paula_sum = load_file('Paula_SUM_LIST.csv')
print(f'Columns: {list(paula_sum.columns)}')
display(paula_sum.head(3))

─── Loading Paula_SUM_LIST ─────────────────────────────────────────────
  ✓  Loaded  Paula_SUM_LIST.csv                            →   26,087 rows × 4 cols
Columns: ['ingredient_name', 'rating', 'functions', 'link']


,ingredient_name,rating,functions,link
0,3-O Ethyl Ascorbic Acid,BEST,This potent antioxidant is a highly stable form of vitamin C that improves s...,https://www.paulaschoice.com/ingredient-dictionary/ingredient-3-o-ethyl-asco...
1,Acai,BEST,"Pronounced ""ah-sigh-ee"", this small berry has a deep purple color and is a r...",https://www.paulaschoice.com/ingredient-dictionary/ingredient-acai.html?csor...
2,Acerola Fruit Extract,BEST,This fruit extract is a potent source of antioxidants and hydrating polysacc...,https://www.paulaschoice.com/ingredient-dictionary/ingredient-acerola-fruit-...


In [82]:
paula_sum_clean = basic_clean(paula_sum)

# Numeric conversion
for col in paula_sum_clean.columns:
    if any(k in col.lower() for k in ['price', 'rating', 'score', 'count', 'num']):
        paula_sum_clean[col] = pd.to_numeric(paula_sum_clean[col], errors='coerce')

quick_report(paula_sum_clean, "Paula SUM LIST — cleaned")
save(paula_sum_clean, '11_paula_sumlist_clean.csv')


──────────────────────────────────────────────────────────────────────
  📋  Paula SUM LIST — cleaned
  Rows : 26,087   Cols : 4
  Missing values:
    rating                                    26,087 (100.0%)
    functions                                     15 (0.1%)
    link                                      23,675 (90.8%)
──────────────────────────────────────────────────────────────────────


,ingredient_name,rating,functions,link
0,3-O Ethyl Ascorbic Acid,NaN,This potent antioxidant is a highly stable form of vitamin C that improves s...,https://www.paulaschoice.com/ingredient-dictionary/ingredient-3-o-ethyl-asco...
1,Acai,NaN,"Pronounced ""ah-sigh-ee"", this small berry has a deep purple color and is a r...",https://www.paulaschoice.com/ingredient-dictionary/ingredient-acai.html?csor...
2,Acerola Fruit Extract,NaN,This fruit extract is a potent source of antioxidants and hydrating polysacc...,https://www.paulaschoice.com/ingredient-dictionary/ingredient-acerola-fruit-...


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\11_paula_sumlist_clean.csv  [26,087 rows × 4 cols]


---
## 🧬 Section 12 — Paula's Choice — Embeddings
**Source:** `Paula_embedding_SUMLIST_before_422` (15.6 MB — largest file!)  
**Output:** `12_paula_embeddings_clean.csv`

This is likely a high-dimensional embedding or feature matrix — we clean metadata columns and verify numeric embedding columns.

In [83]:
print('─── Loading Paula_embedding_SUMLIST_before_422 ─────────────────────────')
print('  ℹ  Large file — this may take a moment...')
paula_emb = load_file('Paula_embedding_SUMLIST_before_422.csv')
print(f'Columns (first 15): {list(paula_emb.columns[:15])}')

─── Loading Paula_embedding_SUMLIST_before_422 ─────────────────────────
  ℹ  Large file — this may take a moment...
  ✓  Loaded  Paula_embedding_SUMLIST_before_422.csv        →   26,074 rows × 11 cols
Columns (first 15): ['ingredient_name', 'rating', 'functions', 'link', 'benefits', 'categories', 'glance', 'references', 'all', 'description', 'combined_text']


In [84]:
paula_emb_clean = basic_clean(paula_emb)

# Separate metadata columns from numeric embedding columns
metadata_keywords = ['id', 'name', 'brand', 'category', 'product', 'label', 'type']
meta_cols = [c for c in paula_emb_clean.columns
             if any(k in c.lower() for k in metadata_keywords)]
emb_cols  = [c for c in paula_emb_clean.columns if c not in meta_cols]

print(f'  Metadata columns  : {len(meta_cols)} → {meta_cols[:8]}')
print(f'  Embedding columns : {len(emb_cols)}  (showing first 5: {emb_cols[:5]})')

# Ensure embedding values are float
for col in emb_cols:
    paula_emb_clean[col] = pd.to_numeric(paula_emb_clean[col], errors='coerce')

# Drop rows where all embedding values are NaN
before = len(paula_emb_clean)
paula_emb_clean = paula_emb_clean.dropna(subset=emb_cols, how='all')
print(f'  Dropped all-NaN embedding rows: {before - len(paula_emb_clean):,}')

quick_report(paula_emb_clean[meta_cols + emb_cols[:5]], 'Paula Embeddings — metadata + first 5 dims')
save(paula_emb_clean, '12_paula_embeddings_clean.csv')

  Metadata columns  : 1 → ['ingredient_name']
  Embedding columns : 10  (showing first 5: ['rating', 'functions', 'link', 'benefits', 'categories'])
  Dropped all-NaN embedding rows: 26,074

──────────────────────────────────────────────────────────────────────
  📋  Paula Embeddings — metadata + first 5 dims
  Rows : 0   Cols : 6
  No missing values.
──────────────────────────────────────────────────────────────────────


,ingredient_name,rating,functions,link,benefits,categories


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\12_paula_embeddings_clean.csv  [0 rows × 11 cols]


---
## 🔄 Section 13 — Pre-Alternatives
**Source:** `pre_alternatives` (53 KB)  
**Output:** `13_alternatives_clean.csv`

In [85]:
print('─── Loading pre_alternatives ───────────────────────────────────────────')
alternatives = load_file('pre_alternatives.csv')
alternatives_clean = basic_clean(alternatives)

print(f'Columns: {list(alternatives_clean.columns)}')
display(alternatives_clean.head(5))

save(alternatives_clean, '13_alternatives_clean.csv')

─── Loading pre_alternatives ───────────────────────────────────────────
  ✓  Loaded  pre_alternatives.csv                          →    1,334 rows × 2 cols
Columns: ['component1', 'component2']


,component1,component2
0,"1,2-HEXANEDIOL",PENTYLENE GLYCOL
1,"1,2-HEXANEDIOL",CAPRYLYL GLYCOL
2,"2-BROMO-2-NITROPROPANE-1,3-DIOL",PHENOXYETHANOL
3,"2-BROMO-2-NITROPROPANE-1,3-DIOL",POTASSIUM SORBATE
4,"2-BROMO-2-NITROPROPANE-1,3-DIOL",CAPRYLYL GLYCOL


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\13_alternatives_clean.csv  [1,334 rows × 2 cols]


---
## 🗄️ Section 14 — Datasheet (Master)
**Source:** `datasheet` (11.9 MB — second largest file)  
**Output:** `14_datasheet_clean.csv`

This is the largest structured file — likely a comprehensive product/ingredient database.

In [86]:
print('─── Loading datasheet ──────────────────────────────────────────────────')
print('  ℹ  Large file — this may take a moment...')
datasheet = load_file('datasheet.csv')
print(f'Columns: {list(datasheet.columns)}')

─── Loading datasheet ──────────────────────────────────────────────────
  ℹ  Large file — this may take a moment...
  ✓  Loaded  datasheet.csv                                 →   19,050 rows × 6 cols
Columns: ['brand', 'name', 'type', 'country', 'ingridients', 'afterUse']


In [87]:
datasheet_clean = basic_clean(datasheet)

# Auto-detect and convert numeric columns
for col in datasheet_clean.select_dtypes(include='object').columns:
    # Attempt numeric conversion only for columns that look numeric
    sample = datasheet_clean[col].dropna().head(20)
    converted = pd.to_numeric(sample.str.replace(r'[\$,£€%]', '', regex=True), errors='coerce')
    if converted.notna().sum() / max(len(sample), 1) > 0.8:  # 80%+ are numeric
        datasheet_clean[col] = datasheet_clean[col].str.replace(r'[\$,£€%]', '', regex=True)
        datasheet_clean[col] = pd.to_numeric(datasheet_clean[col], errors='coerce')
        print(f'  ↳ Converted [{col}] to numeric')

quick_report(datasheet_clean, 'Datasheet — cleaned')
save(datasheet_clean, '14_datasheet_clean.csv')

  ✂  Removed 10 duplicate / empty rows  (19,050 → 19,040)

──────────────────────────────────────────────────────────────────────
  📋  Datasheet — cleaned
  Rows : 19,040   Cols : 6
  Missing values:
    type                                          15 (0.1%)
    country                                    1,831 (9.6%)
    ingridients                                  319 (1.7%)
    afteruse                                   1,523 (8.0%)
──────────────────────────────────────────────────────────────────────


,brand,name,type,country,ingridients,afteruse
0,The Ordinary,Glycolic Acid 7% Toning Solution,Toner,Canada,"Water,Glycolic Acid,Rosa Damascena Flower Water,Centaurea Cyanus Flower Wate...","Good For Oily Skin,Skin Texture,Reduces Large Pores,Anti-Aging,Dark Spots,Br..."
1,La Roche-Posay,Toleriane Hydrating Gentle Face Cleanser,Face Cleanser,France,"Water,Glycerin,Pentaerythrityl Tetraethylhexanoate,Propylene Glycol,Ammonium...","Good For Oily Skin,Redness Reducing,Reduces Irritation,Anti-Aging,Acne Fight..."
2,The Ordinary,Niacinamide 10% + Zinc 1%,Facial Treatment,Canada,"Water,Niacinamide,Pentylene Glycol,Zinc PCA,Dimethyl Isosorbide,Tamarindus I...","Good For Oily Skin,Redness Reducing,Acne Fighting,Brightening,Irritating"


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\14_datasheet_clean.csv  [19,040 rows × 6 cols]


---
## 📈 Section 15 — Analysis / Output Files
**Sources:** `output`, `pretty_numbers`, `pretty_regression`  
**Outputs:** `15_output_clean.csv`, `16_pretty_numbers_clean.csv`, `17_pretty_regression_clean.csv`

In [88]:
# ── output.xlsx ─────────────────────────────────────────────────────────────
print('─── Loading output ─────────────────────────────────────────────────────')
output_df = load_file('output.csv')
output_clean = basic_clean(output_df)

for col in output_clean.select_dtypes(include='object').columns:
    sample = output_clean[col].dropna().head(10)
    conv = pd.to_numeric(sample, errors='coerce')
    if conv.notna().sum() / max(len(sample), 1) > 0.8:
        output_clean[col] = pd.to_numeric(output_clean[col], errors='coerce')

quick_report(output_clean, 'Output — cleaned')
save(output_clean, '15_output_clean.csv')

─── Loading output ─────────────────────────────────────────────────────
  ✓  Loaded  output.csv                                    →      931 rows × 19 cols

──────────────────────────────────────────────────────────────────────
  📋  Output — cleaned
  Rows : 931   Cols : 19
  Missing values:
    brand                                         12 (1.3%)
    price                                         14 (1.5%)
    price_sign                                   563 (60.5%)
    currency                                     563 (60.5%)
    description                                   25 (2.7%)
    rating                                       591 (63.5%)
    category                                     424 (45.5%)
──────────────────────────────────────────────────────────────────────


,id,brand,name,price,price_sign,currency,image_link,product_link,website_link,description,rating,category,product_type,tag_list,created_at,updated_at,product_api_url,api_featured_image,product_colors
0,1048,colourpop,Lippie Pencil,5.00,$,CAD,https://cdn.shopify.com/s/files/1/1338/0845/collections/lippie-pencil_grande...,https://colourpop.com/collections/lippie-pencil,https://colourpop.com,Lippie Pencil A long-wearing and high-intensity lip pencil that glides on ea...,NaN,pencil,lip_liner,"['cruelty free', 'Vegan']",2018-07-08T23:45:08.056Z,2018-07-09T00:53:23.301Z,http://makeup-api.herokuapp.com/api/v1/products/1048.json,//s3.amazonaws.com/donovanbailey/products/api_featured_images/000/001/048/or...,"[{'hex_value': '#B28378', 'colour_name': 'BFF Pencil'}, {'hex_value': '#A36B..."
1,1047,colourpop,Blotted Lip,5.50,$,CAD,https://cdn.shopify.com/s/files/1/1338/0845/products/brain-freeze_a_800x1200...,https://colourpop.com/collections/lippie-stix?filter=blotted-lip,https://colourpop.com,Blotted Lip Sheer matte lipstick that creates the perfect popsicle pout! For...,NaN,lipstick,lipstick,"['cruelty free', 'Vegan']",2018-07-08T22:01:20.178Z,2018-07-09T00:53:23.287Z,http://makeup-api.herokuapp.com/api/v1/products/1047.json,//s3.amazonaws.com/donovanbailey/products/api_featured_images/000/001/047/or...,"[{'hex_value': '#b72227', 'colour_name': ""Bee's Knees""}, {'hex_value': '#BB6..."
2,1046,colourpop,Lippie Stix,5.50,$,CAD,https://cdn.shopify.com/s/files/1/1338/0845/collections/blottedlip-lippie-st...,https://colourpop.com/collections/lippie-stix,https://colourpop.com,"Lippie Stix Formula contains Vitamin E, Mango, Avocado, and Shea butter for ...",NaN,lipstick,lipstick,"['cruelty free', 'Vegan']",2018-07-08T21:47:49.858Z,2018-07-09T00:53:23.274Z,http://makeup-api.herokuapp.com/api/v1/products/1046.json,//s3.amazonaws.com/donovanbailey/products/api_featured_images/000/001/046/or...,"[{'hex_value': '#F2DEC3', 'colour_name': 'Fair 05'}, {'hex_value': '#793C36'..."


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\15_output_clean.csv  [931 rows × 19 cols]


In [89]:
# ── pretty_numbers.xlsx ──────────────────────────────────────────────────────
print('─── Loading pretty_numbers ─────────────────────────────────────────────')
pretty_num = load_file('pretty_numbers.csv')
pretty_num_clean = basic_clean(pretty_num)

# All non-string columns should be numeric
for col in pretty_num_clean.columns:
    pretty_num_clean[col] = pd.to_numeric(pretty_num_clean[col], errors='coerce')

quick_report(pretty_num_clean, 'Pretty Numbers — cleaned')
save(pretty_num_clean, '16_pretty_numbers_clean.csv')

─── Loading pretty_numbers ─────────────────────────────────────────────
  ✓  Loaded  pretty_numbers.csv                            →    1,689 rows × 47 cols

──────────────────────────────────────────────────────────────────────
  📋  Pretty Numbers — cleaned
  Rows : 1,689   Cols : 47
  Missing values:
    size                                         226 (13.4%)
    price_per_ounce                              226 (13.4%)
──────────────────────────────────────────────────────────────────────


,unnamed_0,price,n_of_reviews,n_of_loves,review_score,size,clean_product,category_anti_aging,category_bb__cc_cream,category_bath__shower,category_beauty_supplements,category_blemish__acne_treatments,category_blotting_papers,category_body_lotions__body_oils,category_cellulite__stretch_marks,category_decollete__neck_creams,category_exfoliators,category_eye_creams__treatments,category_eye_masks,category_face_masks,category_face_oils,category_face_primer,category_face_serums,category_face_sunscreen,category_face_wash__cleansers,category_facial_peels,category_foundation,category_hair_oil,category_highlighter,category_holistic_wellness,category_mini_size,category_mists__essences,category_moisturizer__treatments,category_moisturizers,category_night_creams,category_setting_spray__powder,category_sheet_masks,category_skincare,category_tinted_moisturizer,category_toners,category_tools,category_value__gift_sets,reviews_to_loves_ratio,return_on_reviews,price_per_ounce,reviews_to_loves_log,return_on_reviews_log
0,0,68.00,1000,136008,4.21,1.69,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0.74,0.42,40.24,-0.30,-0.87
1,1,175.00,493,61648,4.10,1.00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0.80,0.83,175.00,-0.22,-0.19
2,2,39.00,2000,188389,4.04,1.08,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.06,0.20,36.11,0.06,-1.61


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\16_pretty_numbers_clean.csv  [1,689 rows × 47 cols]


In [90]:
# ── pretty_regression.xlsx ───────────────────────────────────────────────────
print('─── Loading pretty_regression ──────────────────────────────────────────')
pretty_reg = load_file('pretty_regression.csv')
pretty_reg_clean = basic_clean(pretty_reg)

for col in pretty_reg_clean.select_dtypes(include='object').columns:
    sample = pretty_reg_clean[col].dropna().head(10)
    conv = pd.to_numeric(sample, errors='coerce')
    if conv.notna().sum() / max(len(sample), 1) > 0.8:
        pretty_reg_clean[col] = pd.to_numeric(pretty_reg_clean[col], errors='coerce')

quick_report(pretty_reg_clean, 'Pretty Regression — cleaned')
save(pretty_reg_clean, '17_pretty_regression_clean.csv')

─── Loading pretty_regression ──────────────────────────────────────────
  ✓  Loaded  pretty_regression.csv                         →    1,689 rows × 10 cols

──────────────────────────────────────────────────────────────────────
  📋  Pretty Regression — cleaned
  Rows : 1,689   Cols : 10
  Missing values:
    size                                         226 (13.4%)
    price_per_ounce                              226 (13.4%)
──────────────────────────────────────────────────────────────────────


,unnamed_0,price,n_of_reviews,n_of_loves,review_score,size,clean_product,return_on_reviews,reviews_to_loves_ratio,price_per_ounce
0,0,68.00,1000,136008,4.21,1.69,1,0.42,0.74,40.24
1,1,175.00,493,61648,4.10,1.00,0,0.83,0.80,175.00
2,2,39.00,2000,188389,4.04,1.08,0,0.20,1.06,36.11


  💾  Saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned\17_pretty_regression_clean.csv  [1,689 rows × 10 cols]


---
## ✅ Section 16 — Summary Report

In [93]:
import os

def safe_read_csv_header(path):
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
    for enc in encodings:
        try:
            return pd.read_csv(path, nrows=1, encoding=enc), enc
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, nrows=1, encoding="latin1"), "latin1"

def safe_count_rows(path):
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
    for enc in encodings:
        try:
            with open(path, "r", encoding=enc, errors="ignore") as file:
                return sum(1 for _ in file) - 1
        except UnicodeDecodeError:
            continue
    return 0

print("=" * 75)
print("  🌸  GlowAI Data Cleaning — SUMMARY")
print("=" * 75)
print(f'{"File":<40} {"Rows":>10} {"Cols":>6} {"Size (KB)":>10} {"Encoding":>10}')
print("-" * 75)

total_rows = 0
output_files = sorted(OUTPUT_DIR.glob("*.csv"))

for f in output_files:
    df, enc = safe_read_csv_header(f)
    rows = safe_count_rows(f)
    cols = len(df.columns)
    size_kb = os.path.getsize(f) / 1024
    total_rows += rows

    print(f"{f.name:<40} {rows:>10,} {cols:>6} {size_kb:>10.1f} {enc:>10}")

print("=" * 75)
print(f'{"TOTAL":<40} {total_rows:>10,}')
print("=" * 75)
print(f"\nOutput directory: {OUTPUT_DIR.resolve()}")
print("\n✅ All files cleaned and saved successfully!")

  🌸  GlowAI Data Cleaning — SUMMARY
File                                           Rows   Cols  Size (KB)   Encoding
---------------------------------------------------------------------------
01_reviews_clean.csv                      1,107,410     19   506888.4      utf-8
02_skincare_products_clean.csv                3,474     49      646.0      utf-8
03_product_urls_clean.csv                         4      6        0.9      utf-8
04_popularity_rankings_clean.csv                400      8       22.2      utf-8
05_product_info_clean.csv                     8,494     27     7772.2      utf-8
06_productlist_clean.csv                        237      7      202.9      utf-8
07_sephora_clean.csv                         63,111     20     8037.8      utf-8
08_brands_clean.csv                             100      3        1.8      utf-8
09_makeup_clean.csv                           3,157     19     1036.8      utf-8
10_ingredients_clean.csv                     66,417      2     4324.0      utf

---
## 🗺️ Data Map — What Each File Contains

| Cleaned File | Description | Use in GlowAI |
|---|---|---|
| `01_reviews_clean.csv` | All user reviews (combined chunks) | Sentiment analysis, rating model |
| `02_skincare_products_clean.csv` | Skincare products across all sub-categories | Core skincare recommendation engine |
| `03_product_urls_clean.csv` | Product URLs for all categories | Link products to detail pages |
| `04_popularity_rankings_clean.csv` | Rankings by price, loves, reviews, score | Trending / bestseller features |
| `05_product_info_clean.csv` | Detailed product metadata | Product detail pages |
| `06_productlist_clean.csv` | Flat product catalogue | Search & browse |
| `07_sephora_clean.csv` | Sephora-specific product data | Sephora product integration |
| `08_brands_clean.csv` | Brand catalogue | Brand filter feature |
| `09_makeup_clean.csv` | Makeup products (from JSON) | Makeup recommendation engine |
| `10_ingredients_clean.csv` | Binary ingredient matrix | Ingredient sensitivity filter |
| `11_paula_sumlist_clean.csv` | Paula's Choice product list | Paula brand integration |
| `12_paula_embeddings_clean.csv` | Product embedding vectors | Similarity search / ML model |
| `13_alternatives_clean.csv` | Product alternatives mapping | 'You might also like' feature |
| `14_datasheet_clean.csv` | Master product/ingredient database | Core lookup table |
| `15_output_clean.csv` | Processed model output | Analysis results |
| `16_pretty_numbers_clean.csv` | Summary statistics | Dashboard metrics |
| `17_pretty_regression_clean.csv` | Regression analysis results | Model coefficients |